In [ ]:
"""
MAVIA ADAPTIVE ENGINE — SIMULATED ENVIRONMENT (v2)
===================================================
Rebuilt to match the finalized pipeline doc:

  - The DQN's action = (which DAG node to focus on) x (which Bloom tier to
    attempt: Remember / Understand / Apply). Choosing a tier HIGHER than the
    learner's current tier on that node = an "escalate" attempt. Choosing the
    SAME tier = "stay/remediate normally". Choosing LOWER = "de-escalate/
    remediate down". The agent is free to choose any of these -- the
    0.70 / 0.40 thresholds are NOT hard rules blocking it, they're just used
    to compute the reward signal (and included as a state feature so the
    agent can learn to respect them).

  - Prerequisite gating IS a hard rule (per Box 1: "Verify prerequisite
    mastery before allowing access") -- the agent cannot select a node whose
    prerequisites aren't mastered yet. This is enforced with an action mask.

  - Reward is 3-valued, matching "RL Agent Update" in the doc:
        Positive : escalation succeeded, or node fully mastered
        Neutral  : stagnation (tier unchanged)
        Negative : de-escalation, or prerequisite remediation triggered
"""

import numpy as np

# ---------------------------------------------------------------------------
# SYNTHETIC DAG
# A small hand-built prerequisite graph. Replace with your real DAG later --
# the environment only needs a dict of {node: [prerequisite_node_ids]}.
# ---------------------------------------------------------------------------
NODE_NAMES = ["intro", "basics_a", "basics_b", "intermediate", "advanced"]
N_NODES = len(NODE_NAMES)
PREREQS = {
    0: [],        # intro has no prerequisites
    1: [0],       # basics_a needs intro
    2: [0],       # basics_b needs intro
    3: [1, 2],    # intermediate needs both basics_a and basics_b
    4: [3],       # advanced needs intermediate
}

N_TIERS = 3  # Remember(0) / Understand(1) / Apply(2)
ESCALATE_THRESHOLD = 0.70
DEESCALATE_THRESHOLD = 0.40
NODE_MASTERED_THRESHOLD = 0.90
PREREQ_MASTERY_REQUIRED = 0.70  # how much prereq mastery unlocks a node


class MaviaEnv:
    """One simulated learner moving through the DAG."""

    def __init__(self):
        self.mastery = None      # per-node mastery estimate (what BKT tracks)
        self.tier = None         # per-node current Bloom tier (0/1/2)
        self.already_mastered = None  # tracks which nodes have already paid out the mastery bonus
        self.reset()

    def reset(self):
        self.mastery = np.random.uniform(0.05, 0.15, size=N_NODES)
        self.mastery[PREREQS_ROOT_NODES()] = np.random.uniform(0.15, 0.3)  # root nodes start slightly warmer
        self.tier = np.zeros(N_NODES, dtype=np.int64)
        self.already_mastered = np.zeros(N_NODES, dtype=bool)
        return self._get_state()

    # -----------------------------------------------------------------
    # STATE
    # -----------------------------------------------------------------
    def _prereqs_satisfied(self, node):
        return all(self.mastery[p] >= PREREQ_MASTERY_REQUIRED for p in PREREQS[node])

    def valid_action_mask(self):
        """
        Returns a boolean array over the full (node, tier) action space:
        True = allowed. A node is selectable only if its prereqs are met.
        Any tier is allowed on a selectable node (agent chooses escalate/
        stay/de-escalate itself).
        """
        mask = np.zeros(N_NODES * N_TIERS, dtype=bool)
        for node in range(N_NODES):
            if self._prereqs_satisfied(node):
                for tier in range(N_TIERS):
                    mask[encode_action(node, tier)] = True
        return mask

    def _get_state(self):
        prereq_flags = np.array([1.0 if self._prereqs_satisfied(n) else 0.0
                                  for n in range(N_NODES)], dtype=np.float32)
        tier_norm = self.tier.astype(np.float32) / (N_TIERS - 1)
        return np.concatenate([self.mastery.astype(np.float32), tier_norm, prereq_flags])

    # -----------------------------------------------------------------
    # STEP
    # -----------------------------------------------------------------
    def step(self, action_idx):
        node, attempted_tier = decode_action(action_idx)

        if not self._prereqs_satisfied(node):
            # Shouldn't happen if the mask is respected, but guard anyway --
            # this is the "prerequisite remediation" trigger from the doc.
            next_node = self._weakest_prereq(node)
            return self._get_state(), -1.0, False, {
                "event": "prerequisite_remediation", "redirected_to": next_node
            }

        prior_tier = self.tier[node]
        prior_mastery = self.mastery[node]

        # simulate answering a question at the attempted tier
        p_correct = self._simulate_answer(node, attempted_tier)
        correct = np.random.rand() < p_correct

        # --- REAL BKT UPDATE (Box 5): uses previous mastery, guess, slip, learn ---
        p_guess = 0.2
        p_slip = 0.1
        p_learn = 0.15

        if correct:
            numerator = prior_mastery * (1 - p_slip)
            denom = numerator + (1 - prior_mastery) * p_guess
        else:
            numerator = prior_mastery * p_slip
            denom = numerator + (1 - prior_mastery) * (1 - p_guess)
        posterior = numerator / max(denom, 1e-6)
        new_mastery = posterior + (1 - posterior) * p_learn
        self.mastery[node] = new_mastery
        # -----------------------------------------------------------------

        # --- Box 6/7: tier recomputation + routing outcome (drives reward only) ---
        if new_mastery > ESCALATE_THRESHOLD:
            new_tier = min(prior_tier + 1, N_TIERS - 1) if correct else prior_tier
        elif new_mastery < DEESCALATE_THRESHOLD:
            new_tier = max(prior_tier - 1, 0)
        else:
            new_tier = prior_tier
        self.tier[node] = new_tier

        node_mastered_now = (new_mastery >= NODE_MASTERED_THRESHOLD and new_tier == N_TIERS - 1
                             and not self.already_mastered[node])
        if node_mastered_now:
            self.already_mastered[node] = True

        if node_mastered_now:
            reward = 1.0
            event = "node_mastered"
        elif new_tier > prior_tier:
            reward = 1.0
            event = "escalation"
        elif new_tier < prior_tier:
            reward = -1.0
            event = "deescalation"
        elif self.already_mastered[node]:
            reward = -0.2  # drilling content already mastered wastes a turn -- discourage it
            event = "wasted_repetition"
        else:
            reward = 0.0
            event = "stagnation"

        done = bool(np.all(self.mastery >= NODE_MASTERED_THRESHOLD))
        return self._get_state(), reward, done, {"event": event, "correct": correct}

    def _simulate_answer(self, node, attempted_tier):
        """
        How likely the simulated learner is to answer correctly, given their
        real mastery and how far the attempted tier is from what they're
        actually ready for. This is the "ground truth" the DQN never sees
        directly -- only outcomes (correct/incorrect) filter through.
        """
        base = self.mastery[node]
        tier_gap = attempted_tier - self.tier[node]
        # attempting a harder tier than current level reduces success chance
        penalty = 0.25 * max(tier_gap, 0)
        p = np.clip(base - penalty, 0.02, 0.98)
        return p

    def _weakest_prereq(self, node):
        prereqs = PREREQS[node]
        if not prereqs:
            return node
        return min(prereqs, key=lambda p: self.mastery[p])


def PREREQS_ROOT_NODES():
    return [n for n, p in PREREQS.items() if not p]


def encode_action(node, tier):
    return node * N_TIERS + tier


def decode_action(action_idx):
    node = action_idx // N_TIERS
    tier = action_idx % N_TIERS
    return node, tier


def n_actions():
    return N_NODES * N_TIERS


def state_dim():
    return N_NODES * 3  # mastery + tier + prereq_flag per node

In [ ]:
"""
DQN AGENT
=========
Standard DQN: a small MLP approximates Q(state, action) -- "how good is it
to serve this exercise given this learner's current mastery state".

Two networks are used (policy_net + target_net) because training a single
network against its own shifting predictions is unstable -- this is the
one non-obvious trick that makes DQN work at all. target_net is a slow-moving
copy of policy_net, updated every N steps.
"""

import random
from collections import deque

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


class QNetwork(nn.Module):
    def __init__(self, state_dim, n_actions, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    """Stores past experience so we train on random batches, not just the
    most recent transition -- breaks harmful correlation between consecutive
    samples."""

    def __init__(self, capacity=20000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done, next_mask):
        self.buffer.append((state, action, reward, next_state, done, next_mask))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones, next_masks = zip(*batch)
        return (
            np.array(states, dtype=np.float32),
            np.array(actions, dtype=np.int64),
            np.array(rewards, dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(dones, dtype=np.float32),
            np.array(next_masks, dtype=bool),
        )

    def __len__(self):
        return len(self.buffer)


class DQNAgent:
    def __init__(self, state_dim, n_actions, lr=1e-3, gamma=0.95,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=2000):
        self.n_actions = n_actions
        self.gamma = gamma

        self.policy_net = QNetwork(state_dim, n_actions)
        self.target_net = QNetwork(state_dim, n_actions)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.replay = ReplayBuffer()

        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.steps_done = 0

    def epsilon(self):
        # exploration rate decays from ~100% random to ~5% random over training
        return self.epsilon_end + (self.epsilon_start - self.epsilon_end) * \
            np.exp(-1.0 * self.steps_done / self.epsilon_decay)

    def select_action(self, state, explore=True, valid_mask=None):
        self.steps_done += 1
        if explore and random.random() < self.epsilon():
            if valid_mask is not None:
                valid_indices = np.flatnonzero(valid_mask)
                return int(np.random.choice(valid_indices))
            return random.randrange(self.n_actions)
        with torch.no_grad():
            state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
            q_values = self.policy_net(state_t).squeeze(0).numpy()
            if valid_mask is not None:
                q_values = np.where(valid_mask, q_values, -1e9)  # mask out illegal actions
            return int(np.argmax(q_values))

    def train_step(self, batch_size=64):
        if len(self.replay) < batch_size:
            return None  # not enough data yet

        states, actions, rewards, next_states, dones, next_masks = self.replay.sample(batch_size)

        states_t = torch.tensor(states)
        actions_t = torch.tensor(actions).unsqueeze(1)
        rewards_t = torch.tensor(rewards)
        next_states_t = torch.tensor(next_states)
        dones_t = torch.tensor(dones)
        next_masks_t = torch.tensor(next_masks)

        # Q(s,a) for the actions actually taken
        q_values = self.policy_net(states_t).gather(1, actions_t).squeeze(1)

        # target: r + gamma * max_a' Q_target(s', a'), zeroed out if episode ended
        with torch.no_grad():
            next_q_all = self.target_net(next_states_t)
            next_q_all = next_q_all.masked_fill(~next_masks_t, -1e9)  # illegal actions can't be chosen
            next_q_values = next_q_all.max(1)[0]
            target = rewards_t + self.gamma * next_q_values * (1 - dones_t)

        loss = nn.functional.mse_loss(q_values, target)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()

    def update_target(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())

    def save(self, path):
        torch.save(self.policy_net.state_dict(), path)

    def load(self, path):
        self.policy_net.load_state_dict(torch.load(path))
        self.target_net.load_state_dict(self.policy_net.state_dict())

In [ ]:
import sys
import numpy as np

N_EPISODES = 3000
MAX_STEPS_PER_EPISODE = 80
TARGET_UPDATE_EVERY = 200
PRINT_EVERY = 150
CHECKPOINT_PATH = "dqn_policy_v2.pt"


def heuristic_action(env):
    """Baseline: always attempt the lowest-mastery node that's unlocked,
    at the tier matching its literal threshold band (i.e. blindly follow
    the fixed thresholds instead of learning anything)."""
    mask = env.valid_action_mask()
    best_node, best_mastery = None, 2.0
    for node in range(len(env.mastery)):
        if env._prereqs_satisfied(node) and env.mastery[node] < best_mastery:
            best_node, best_mastery = node, env.mastery[node]
    m = env.mastery[best_node]
    if m > 0.70:
        tier = min(env.tier[best_node] + 1, N_TIERS - 1)
    elif m < 0.40:
        tier = max(env.tier[best_node] - 1, 0)
    else:
        tier = env.tier[best_node]
    # encode_action is defined in the environment cell
    return encode_action(best_node, tier)


def run_episode(env, policy_fn, max_steps=MAX_STEPS_PER_EPISODE):
    state = env.reset()
    total_reward = 0.0
    for t in range(max_steps):
        action = policy_fn(env, state)
        state, reward, done, info = env.step(action)
        total_reward += reward
        if done:
            break
    return total_reward, np.mean(env.mastery)


def main(n_episodes=N_EPISODES, resume=False):
    env = MaviaEnv()
    agent = DQNAgent(state_dim=state_dim(), n_actions=n_actions())
    if resume:
        agent.load(CHECKPOINT_PATH)
        print(f"Resumed from {CHECKPOINT_PATH}")

    episode_rewards = []

    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0.0

        for t in range(MAX_STEPS_PER_EPISODE):
            mask = env.valid_action_mask()
            action = agent.select_action(state, valid_mask=mask)
            next_state, reward, done, info = env.step(action)
            next_mask = env.valid_action_mask()
            agent.replay.push(state, action, reward, next_state, done, next_mask)

            agent.train_step()

            state = next_state
            total_reward += reward

            if agent.steps_done % TARGET_UPDATE_EVERY == 0:
                agent.update_target()

            if done:
                break

        episode_rewards.append(total_reward)

        if (episode + 1) % PRINT_EVERY == 0:
            avg_reward = np.mean(episode_rewards[-PRINT_EVERY:])
            print(f"Episode {episode+1}/{n_episodes} | "
                  f"avg reward (last {PRINT_EVERY}): {avg_reward:.2f} | "
                  f"epsilon: {agent.epsilon():.3f}")

    agent.save(CHECKPOINT_PATH)
    print(f"\nSaved trained weights to {CHECKPOINT_PATH}")

    import json
    history_path = "reward_history.json"
    existing = []
    if resume:
        try:
            with open(history_path) as f:
                existing = json.load(f)
        except FileNotFoundError:
            pass
    with open(history_path, "w") as f:
        json.dump(existing + episode_rewards, f)
    print(f"Saved reward history to {history_path}")

    # --- Evaluation: three-way baseline comparison ---
    print("\n--- Evaluation: trained DQN vs heuristic vs random ---")
    print(f"(avg final mean-mastery across all nodes after {MAX_STEPS_PER_EPISODE} steps, higher is better)\n")

    def trained_policy(e, s):
        mask = e.valid_action_mask()
        return agent.select_action(s, explore=False, valid_mask=mask)

    def random_policy(e, s):
        mask = e.valid_action_mask()
        valid_indices = np.flatnonzero(mask)
        return int(np.random.choice(valid_indices))

    def heuristic_policy(e, s):
        return heuristic_action(e)

    for label, policy_fn in [
        ("trained DQN", trained_policy),
        ("heuristic (follow thresholds literally)", heuristic_policy),
        ("random (valid actions only)", random_policy),
    ]:
        rewards, masteries = [], []
        for _ in range(200):
            r, m = run_episode(env, policy_fn)
            rewards.append(r)
            masteries.append(m)
        print(f"{label:42s} | avg reward = {np.mean(rewards):6.2f} | avg final mastery = {np.mean(masteries):.3f}")


if __name__ == "__main__":
    main()

Episode 150/3000 | avg reward (last 150): 5.03 | epsilon: 0.053
Episode 300/3000 | avg reward (last 150): 7.73 | epsilon: 0.050
Episode 450/3000 | avg reward (last 150): 7.49 | epsilon: 0.050
Episode 600/3000 | avg reward (last 150): 7.89 | epsilon: 0.050
Episode 750/3000 | avg reward (last 150): 7.82 | epsilon: 0.050
Episode 900/3000 | avg reward (last 150): 8.07 | epsilon: 0.050
Episode 1050/3000 | avg reward (last 150): 7.81 | epsilon: 0.050
Episode 1200/3000 | avg reward (last 150): 7.93 | epsilon: 0.050
Episode 1350/3000 | avg reward (last 150): 7.73 | epsilon: 0.050
Episode 1500/3000 | avg reward (last 150): 7.60 | epsilon: 0.050
Episode 1650/3000 | avg reward (last 150): 7.71 | epsilon: 0.050
Episode 1800/3000 | avg reward (last 150): 7.96 | epsilon: 0.050
Episode 1950/3000 | avg reward (last 150): 7.75 | epsilon: 0.050
Episode 2100/3000 | avg reward (last 150): 7.80 | epsilon: 0.050
Episode 2250/3000 | avg reward (last 150): 7.97 | epsilon: 0.050
Episode 2400/3000 | avg reward 

In [ ]:
"""
INDEPENDENT SIMULATOR VALIDATION
================================
This does NOT test the DQN. It tests whether environment_v2.py itself
behaves the way a learning environment should, independent of any agent.
This is the "validate the simulator independently" rubric item -- you're
proving the simulator's internal logic is trustworthy before trusting any
results trained against it.

Run: python validate_simulator.py
"""
import numpy as np

PASS, FAIL = "PASS", "FAIL"
results = []


def check(name, condition):
    results.append((name, PASS if condition else FAIL))
    print(f"[{PASS if condition else FAIL}] {name}")


# --- Test 1: high mastery should answer correctly far more often than low mastery ---
env = MaviaEnv()
env.reset()
node = 0
env.mastery[node] = 0.9
env.tier[node] = 2
high_p = env._simulate_answer(node, attempted_tier=2)

env.mastery[node] = 0.1
env.tier[node] = 0
low_p = env._simulate_answer(node, attempted_tier=0)

check("High mastery learner has higher correctness probability than low mastery learner",
      high_p > low_p)

# --- Test 2: prerequisite gating blocks locked nodes ---
env = MaviaEnv()
env.reset()
env.mastery[:] = 0.0  # nobody has mastered anything
mask = env.valid_action_mask()
node3_actions = [encode_action(3, t) for t in range(N_TIERS)]  # node 3 needs nodes 1 & 2
check("Node with unmet prerequisites is locked out of the action mask",
      not any(mask[a] for a in node3_actions))

env.mastery[1] = 0.9
env.mastery[2] = 0.9  # now satisfy node 3's prerequisites
mask = env.valid_action_mask()
check("Node becomes selectable once its prerequisites are satisfied",
      all(mask[a] for a in node3_actions))

# --- Test 3: root node with no prerequisites is always selectable ---
env = MaviaEnv()
env.reset()
env.mastery[:] = 0.0
mask = env.valid_action_mask()
node0_actions = [encode_action(0, t) for t in range(N_TIERS)]
check("Root node (no prerequisites) is selectable even at zero mastery",
      all(mask[a] for a in node0_actions))

# --- Test 4: attempting a tier above current level is harder than an easier tier ---
env = MaviaEnv()
env.reset()
env.mastery[0] = 0.5
env.tier[0] = 0
p_easy = env._simulate_answer(0, attempted_tier=0)
p_hard = env._simulate_answer(0, attempted_tier=2)
check("Attempting a tier above the learner's current level reduces success probability",
      p_hard < p_easy)

# --- Test 5: escalation reward fires only when tier actually increases ---
env = MaviaEnv()
env.reset()
env.mastery[0] = 0.65  # just under escalate threshold
env.tier[0] = 0
np.random.seed(0)
_, reward, _, info = env.step(encode_action(0, 0))
check("Escalation event ('escalation'/'node_mastered') always pays positive reward",
      info["event"] not in ("escalation", "node_mastered") or reward > 0)

# --- Test 6: de-escalation always pays negative reward ---
env = MaviaEnv()
env.reset()
env.mastery[0] = 0.35
env.tier[0] = 2
for _ in range(20):
    np.random.seed(np.random.randint(0, 100000))
    e = MaviaEnv()
    e.reset()
    e.mastery[0] = 0.35
    e.tier[0] = 2
    _, reward, _, info = e.step(encode_action(0, 0))
    if info["event"] == "deescalation":
        check("De-escalation event always pays negative reward", reward < 0)
        break
else:
    print("[SKIP] De-escalation didn't trigger in 20 random tries (not a failure, just unlucky RNG)")

# --- Test 7: node_mastered reward only pays out once per node ---
env = MaviaEnv()
env.reset()
env.mastery[0] = 0.95
env.tier[0] = 2
env.already_mastered[0] = True  # simulate it already having paid out once
_, reward, _, info = env.step(encode_action(0, 2))
check("Drilling an already-mastered node does not pay the mastery bonus again",
      info["event"] != "node_mastered")

# --- Summary ---
n_pass = sum(1 for _, r in results if r == PASS)
print(f"\n{n_pass}/{len(results)} checks passed")
if n_pass < len(results):
    print("Fix failing checks before trusting training results built on this simulator.")

[PASS] High mastery learner has higher correctness probability than low mastery learner
[PASS] Node with unmet prerequisites is locked out of the action mask
[PASS] Node becomes selectable once its prerequisites are satisfied
[PASS] Root node (no prerequisites) is selectable even at zero mastery
[PASS] Attempting a tier above the learner's current level reduces success probability
[PASS] Escalation event ('escalation'/'node_mastered') always pays positive reward
[PASS] De-escalation event always pays negative reward
[PASS] Drilling an already-mastered node does not pay the mastery bonus again

8/8 checks passed


In [ ]:
"""
Generates the reward-vs-episode convergence plot from reward_history.json.
Run this AFTER training (it reads what train_v2.py already saved).
"""
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

with open("reward_history.json") as f:
    rewards = json.load(f)

rewards = np.array(rewards)
window = 50
smoothed = np.convolve(rewards, np.ones(window) / window, mode="valid")

plt.figure(figsize=(9, 5))
plt.plot(rewards, alpha=0.25, color="steelblue", label="raw episode reward")
plt.plot(range(window - 1, len(rewards)), smoothed, color="steelblue", linewidth=2,
         label=f"{window}-episode moving average")
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.title("DQN Training Convergence — MAVIA Adaptive Engine")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("convergence_curve.png", dpi=150)
print("Saved convergence_curve.png")

Saved convergence_curve.png
